# Predict H1N1 and Seasonal Flu Vaccines - EDA

Import libraries 

In [2]:
import pandas as pd
import numpy as np

Load data

In [3]:
X_train = pd.read_csv('../data/training_set_features.csv')
y_train = pd.read_csv('../data/training_set_labels.csv')

In [4]:
df = X_train.merge(y_train, on='respondent_id')

## Data Profiling

In [5]:
# Fast view
df.head(3).T

,0,1,2
respondent_id,0,1,2
h1n1_concern,1.0,3.0,1.0
h1n1_knowledge,0.0,2.0,1.0
behavioral_antiviral_meds,0.0,0.0,0.0
behavioral_avoidance,0.0,1.0,1.0
behavioral_face_mask,0.0,0.0,0.0
behavioral_wash_hands,0.0,1.0,0.0
behavioral_large_gatherings,0.0,0.0,0.0
behavioral_outside_home,1.0,1.0,0.0
behavioral_touch_face,1.0,1.0,0.0


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26707 entries, 0 to 26706
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   respondent_id                26707 non-null  int64  
 1   h1n1_concern                 26615 non-null  float64
 2   h1n1_knowledge               26591 non-null  float64
 3   behavioral_antiviral_meds    26636 non-null  float64
 4   behavioral_avoidance         26499 non-null  float64
 5   behavioral_face_mask         26688 non-null  float64
 6   behavioral_wash_hands        26665 non-null  float64
 7   behavioral_large_gatherings  26620 non-null  float64
 8   behavioral_outside_home      26625 non-null  float64
 9   behavioral_touch_face        26579 non-null  float64
 10  doctor_recc_h1n1             24547 non-null  float64
 11  doctor_recc_seasonal         24547 non-null  float64
 12  chronic_med_condition        25736 non-null  float64
 13  child_under_6_months       

In [7]:
# Descriptive statistics
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
respondent_id,26707.0,NaN,NaN,NaN,13353.0,7709.791156,0.0,6676.5,13353.0,20029.5,26706.0
h1n1_concern,26615.0,NaN,NaN,NaN,1.618486,0.910311,0.0,1.0,2.0,2.0,3.0
h1n1_knowledge,26591.0,NaN,NaN,NaN,1.262532,0.618149,0.0,1.0,1.0,2.0,2.0
behavioral_antiviral_meds,26636.0,NaN,NaN,NaN,0.048844,0.215545,0.0,0.0,0.0,0.0,1.0
behavioral_avoidance,26499.0,NaN,NaN,NaN,0.725612,0.446214,0.0,0.0,1.0,1.0,1.0
behavioral_face_mask,26688.0,NaN,NaN,NaN,0.068982,0.253429,0.0,0.0,0.0,0.0,1.0
behavioral_wash_hands,26665.0,NaN,NaN,NaN,0.825614,0.379448,0.0,1.0,1.0,1.0,1.0
behavioral_large_gatherings,26620.0,NaN,NaN,NaN,0.35864,0.47961,0.0,0.0,0.0,1.0,1.0
behavioral_outside_home,26625.0,NaN,NaN,NaN,0.337315,0.472802,0.0,0.0,0.0,1.0,1.0
behavioral_touch_face,26579.0,NaN,NaN,NaN,0.677264,0.467531,0.0,0.0,1.0,1.0,1.0


In [8]:
# Percentage of missing values
nulls = (df.isnull().sum() / df.shape[0] * 100).sort_values(ascending=False)
print(nulls)

employment_occupation          50.436215
employment_industry            49.912008
health_insurance               45.957989
income_poverty                 16.561201
doctor_recc_seasonal            8.087767
doctor_recc_h1n1                8.087767
rent_or_own                     7.645936
employment_status               5.477965
marital_status                  5.272026
education                       5.268282
chronic_med_condition           3.635751
child_under_6_months            3.070356
health_worker                   3.010447
opinion_seas_sick_from_vacc     2.010709
opinion_seas_risk               1.924589
opinion_seas_vacc_effective     1.729884
opinion_h1n1_sick_from_vacc     1.479013
opinion_h1n1_vacc_effective     1.464036
opinion_h1n1_risk               1.452803
household_children              0.932340
household_adults                0.932340
behavioral_avoidance            0.778822
behavioral_touch_face           0.479275
h1n1_knowledge                  0.434343
h1n1_concern    

### Individual analysis of missing values 

#### `employment_industry` and `employment_occupation`

Estas variables contienen una gran cantidad de valores faltantes, lo que podría estar relacionado con la variable `employment_status`, ya que las personas desempleadas o fuera de la población activa pueden no tener una ocupación o industria asociada.


In [9]:
pd.crosstab(
    df['employment_status'],
    df['employment_industry'].isna()
)

employment_industry,False,True
employment_status,,
Employed,13377,183
Not in Labor Force,0,10231
Unemployed,0,1453


In [10]:
pd.crosstab(
    df['employment_status'],
    df['employment_occupation'].isna()
)

employment_occupation,False,True
employment_status,,
Employed,13237,323
Not in Labor Force,0,10231
Unemployed,0,1453


Most missing values in `employment_industry` and `employment_occupation` correspond to unemployed individuals or people outside the labor force. Therefore, these missing values are likely structural ("Not Applicable") rather than random data omissions.

Solution

In [11]:
df['employment_industry'] = (
    df['employment_industry']
    .fillna('Not_Working')
)

df['employment_occupation'] = (
    df['employment_occupation']
    .fillna('Not_Working')
)

In [12]:
print(df['employment_industry'].isna().sum())
print(df['employment_occupation'].isna().sum())

0
0


#### `health_insurance`   

In [13]:
df['health_insurance'].value_counts(dropna=False)

health_insurance
1.0    12697
NaN    12274
0.0     1736
Name: count, dtype: int64

In [14]:
missing_df = df.isnull().astype(int)

missing_df.corr()['health_insurance'] \
          .sort_values(ascending=False)

health_insurance               1.000000
marital_status                 0.225898
employment_status              0.223742
education                      0.223438
doctor_recc_h1n1               0.221365
doctor_recc_seasonal           0.221365
rent_or_own                    0.194676
child_under_6_months           0.192126
health_worker                  0.186649
chronic_med_condition          0.157246
opinion_seas_sick_from_vacc    0.152659
opinion_seas_risk              0.139328
income_poverty                 0.131229
opinion_h1n1_sick_from_vacc    0.126640
opinion_seas_vacc_effective    0.126587
opinion_h1n1_risk              0.114082
opinion_h1n1_vacc_effective    0.104030
household_adults               0.102071
household_children             0.102071
behavioral_wash_hands          0.008907
behavioral_avoidance           0.007186
behavioral_touch_face          0.000189
h1n1_concern                  -0.000361
behavioral_face_mask          -0.002063
behavioral_outside_home       -0.006363


In [15]:
df['health_insurance'].value_counts(dropna=False)

health_insurance
1.0    12697
NaN    12274
0.0     1736
Name: count, dtype: int64

In [16]:
pd.crosstab(
    df['health_insurance'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
health_insurance,,
0.0,85.253456,14.746544
1.0,68.228715,31.771285
Unknown,88.724132,11.275868


In [17]:
pd.crosstab(
    df['health_insurance'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
health_insurance,,
0.0,77.073733,22.926267
1.0,46.199890,53.800110
Unknown,57.585139,42.414861


In [18]:
df['health_insurance'] = df['health_insurance'].fillna('Unknown') 

In [19]:
df['health_insurance'].value_counts(dropna=False)

health_insurance
1.0        12697
Unknown    12274
0.0         1736
Name: count, dtype: int64

La variable health_insurance presenta un 45.96% de valores faltantes. El análisis de la variable objetivo muestra que los individuos con valores ausentes tienen tasas de vacunación significativamente diferentes tanto de quienes poseen seguro médico como de quienes no lo poseen. Esto indica que la ausencia de información es informativa y no aleatoria. Por tanto, los valores faltantes se recodificaron como una categoría independiente (Unknown) para preservar su capacidad predictiva.

#### `income_poverty`

`income_poverty` es una de las variables que más cuidado merece porque suele ser muy predictiva en problemas relacionados con salud, comportamiento preventivo y acceso a servicios sanitarios.

In [20]:
df['income_poverty'].value_counts(dropna=False)

income_poverty
<= $75,000, Above Poverty    12777
> $75,000                     6810
NaN                           4423
Below Poverty                 2697
Name: count, dtype: int64

Los nulos representan aproximadamente un 16,6% del dataset, una proporción suficientemente grande como para no ignorarla. Veamos que relacion tiene con la vacunacion: 

In [21]:
pd.crosstab(
    df['income_poverty'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
income_poverty,,
"<= $75,000, Above Poverty",79.658762,20.341238
"> $75,000",74.698972,25.301028
Below Poverty,80.867631,19.132369
Unknown,81.098802,18.901198


Observamos que:

- Los ingresos altos tienen una tasa de vacunación superior.
- Los grupos "Below Poverty" y "Unknown" son prácticamente idénticos.
- La diferencia entre "Below Poverty" (19.13%) y "Unknown" (18.90%) es mínima.

In [22]:
pd.crosstab(
    df['income_poverty'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
income_poverty,,
"<= $75,000, Above Poverty",52.328403,47.671597
"> $75,000",50.323054,49.676946
Below Poverty,63.737486,36.262514
Unknown,55.166177,44.833823


Aquí aparecen diferencias más marcadas:

- Los ingresos bajos muestran una menor tasa de vacunación.
- El grupo Unknown queda entre los grupos de ingresos bajos y medios.

La variable income_poverty contiene un 16.56% de valores ausentes. Un análisis de las tasas de vacunación muestra que los encuestados que no aportaron información sobre sus ingresos exhiben un comportamiento similar al de los grupos de menores ingresos, en particular para la vacunación contra el H1N1. 

Sin embargo, la categoría de datos ausentes sigue mostrando patrones distintivos en los resultados de la vacunación estacional. Dado que los valores ausentes pueden contener información socioeconómica en lugar de representar omisiones aleatorias, se conservaron como una categoría separada (`Unknown`) para preservar una posible señal predictiva y evitar la introducción de supuestos mediante la imputación.

In [23]:
df['income_poverty'] = (
    df['income_poverty']
      .fillna('Unknown')
)
df['income_poverty'].value_counts(dropna=False)

income_poverty
<= $75,000, Above Poverty    12777
> $75,000                     6810
Unknown                       4423
Below Poverty                 2697
Name: count, dtype: int64

#### `doctor_recc_h1n1`

Esta caracteristica es la recomendacion del medico para tomar la vacuna contra la gripe H1N1. Veamos si hay correlacion con la aplicacion de la vacuna. 

In [24]:
df['doctor_recc_h1n1'].value_counts(dropna=False)

doctor_recc_h1n1
0.0    19139
1.0     5408
NaN     2160
Name: count, dtype: int64

In [25]:
pd.crosstab(
    df['doctor_recc_h1n1'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
doctor_recc_h1n1,,
0.0,86.362924,13.637076
1.0,46.764053,53.235947
Unknown,91.435185,8.564815


La recomendación médica multiplica aproximadamente por 4 veces la probabilidad de vacunación.

In [26]:
pd.crosstab(
    df['doctor_recc_h1n1'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
doctor_recc_h1n1,,
0.0,57.714614,42.285386
1.0,33.783284,66.216716
Unknown,64.768519,35.231481


La recomandacion médica para la vacuna H1N1, tambien influye en la vacunacion estacional, tambien puede ser que estos recibieran ambas recomendaciones. 

In [27]:
pd.crosstab(
    df['doctor_recc_h1n1'].fillna('Unknown'),
    df['health_insurance'],
    normalize='index'
) * 100

health_insurance,0.0,1.0,Unknown
doctor_recc_h1n1,,,
0.0,7.388056,48.132086,44.479858
1.0,5.232988,58.431953,36.335059
Unknown,1.805556,15.046296,83.148148


Los valores faltantes en `doctor_recc_h1n1` no se distribuyen de forma aleatoria. Los encuestados con información ausente sobre la recomendación médica presentan tasas de vacunación significativamente más bajas y una fuerte asociación con valores faltantes en `health_insurance`, donde más del 83% de los individuos con recomendación médica desconocida también tienen información desconocida sobre su seguro médico.

Este patrón sugiere que la ausencia de datos es informativa y probablemente corresponde a un mecanismo **MNAR (Missing Not At Random)**. Además, los individuos con valores faltantes muestran características distintas de aquellos que sí reportaron haber recibido o no una recomendación médica, por lo que imputarlos como una de las categorías existentes podría introducir sesgo y pérdida de información.

Por esta razón, los valores faltantes fueron conservados como una categoría independiente (`Unknown`), se codificaron con el valor 2.0, con el objetivo de preservar la información contenida en el patrón de ausencia y permitir que el modelo capture posibles diferencias entre los individuos con respuestas conocidas y aquellos con información faltante.


In [28]:
df['doctor_recc_h1n1'] = (
    df['doctor_recc_h1n1']
      .fillna(2)
)
df['doctor_recc_h1n1'].value_counts(dropna=False)

doctor_recc_h1n1
0.0    19139
1.0     5408
2.0     2160
Name: count, dtype: int64

#### `doctor_recc_seasonal`

Esta caracteristica es la recomendacion del medico para tomar la vacuna contra la gripe H1N1. Veamos si hay correlacion con la aplicacion de la vacuna y si tiene los mismos comportamientos que la caracteristica anterior. 

In [29]:
df['doctor_recc_seasonal'].value_counts(dropna=False)

doctor_recc_seasonal
0.0    16453
1.0     8094
NaN     2160
Name: count, dtype: int64

In [30]:
pd.crosstab(
    df['doctor_recc_seasonal'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
doctor_recc_seasonal,,
0.0,65.374096,34.625904
1.0,26.155177,73.844823
Unknown,64.768519,35.231481


La recomendación médica tiene un gran impacto en la aplicacion de la vacuna: `73.8%` vs `34.6%`

In [31]:
pd.crosstab(
    df['doctor_recc_seasonal'].fillna('Unknown'),
    df['health_insurance'],
    normalize='index'
) * 100

health_insurance,0.0,1.0,Unknown
doctor_recc_seasonal,,,
0.0,8.308515,47.711664,43.979821
1.0,4.077094,55.868545,40.054361
Unknown,1.805556,15.046296,83.148148


Aquí vuelve a aparecer el mismo fenómeno. Más del 83% de los individuos con recomendación estacional desconocida también tienen seguro médico desconocido.

In [32]:
pd.crosstab(
    df['seasonal_vaccine'].fillna('Unknown'),
    df['health_insurance'],
    normalize='index'
) * 100

health_insurance,0.0,1.0,Unknown
seasonal_vaccine,,,
0,9.375000,41.101457,49.523543
1,3.200643,54.933655,41.865702


La variable `doctor_recc_seasonal` presenta un 8,09% de valores faltantes. Aunque los encuestados con valores ausentes muestran tasas de vacunación similares a las de aquellos que no recibieron una recomendación médica, un análisis adicional reveló una fuerte asociación entre la ausencia de información en esta variable y la ausencia de información en `health_insurance`.

En concreto, más del 83% de los individuos con una recomendación médica estacional desconocida también presentan un valor desconocido en la variable de seguro médico. Este patrón sugiere que los valores faltantes no se producen completamente al azar y podrían formar parte de un mecanismo de ausencia informativo.

Dado que la ausencia de datos puede contener información relevante para la predicción y con el fin de mantener la consistencia con el tratamiento aplicado a `doctor_recc_h1n1`, los valores faltantes fueron recodificados como una categoría independiente (`Unknown`) y se codificaron con el valor 2.0 con el objetivo de preservar la información contenida en el patrón de ausencia y permitir que el modelo capture posibles diferencias entre los individuos con respuestas conocidas y aquellos con información faltante.


In [33]:
df['doctor_recc_seasonal'] = (
    df['doctor_recc_seasonal']
      .fillna('Unknown')
)
df['doctor_recc_h1n1'].value_counts(dropna=False)

doctor_recc_h1n1
0.0    19139
1.0     5408
2.0     2160
Name: count, dtype: int64

#### `education`

In [ ]:
df['education'].value_counts(dropna=False)

education
College Graduate    10097
Some College         7043
12 Years             5797
< 12 Years           2363
NaN                  1407
Name: count, dtype: int64

In [35]:
pd.crosstab(
    df['education'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
education,,
12 Years,81.524927,18.475073
< 12 Years,83.283961,16.716039
College Graduate,75.408537,24.591463
Some College,79.213403,20.786597
Unknown,81.449893,18.550107


In [36]:
pd.crosstab(
    df['education'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
education,,
12 Years,55.183716,44.816284
< 12 Years,59.881507,40.118493
College Graduate,48.856096,51.143904
Some College,54.806191,45.193809
Unknown,61.478323,38.521677


Aqui se observa un patron: mayor nivel educativo = mayor tasa de vacunacion. Por tanto `education` puede ser una variable predictiva relevante.

In [37]:
missing_df.corr()['education'] \
          .sort_values(ascending=False)

education                      1.000000
marital_status                 0.887874
employment_status              0.863946
child_under_6_months           0.750822
rent_or_own                    0.738230
health_worker                  0.732365
chronic_med_condition          0.620403
opinion_seas_sick_from_vacc    0.591910
opinion_seas_risk              0.550103
opinion_seas_vacc_effective    0.504767
opinion_h1n1_sick_from_vacc    0.498732
income_poverty                 0.491454
opinion_h1n1_risk              0.453232
opinion_h1n1_vacc_effective    0.396872
household_children             0.390444
household_adults               0.390444
health_insurance               0.223438
doctor_recc_h1n1               0.078813
doctor_recc_seasonal           0.078813
h1n1_concern                   0.037626
behavioral_wash_hands          0.024479
behavioral_large_gatherings    0.015932
behavioral_avoidance           0.015333
behavioral_touch_face          0.012757
behavioral_face_mask           0.012566


Las correlaciones con el patron de ausencia son altos con variables socioeconómicas y demográficas, no  directamente relacionadas con la vacunación o el sistema sanitario.

La variable education presenta un 5,27% de valores faltantes. El análisis de las variables objetivo muestra que los individuos con nivel educativo desconocido presentan tasas de vacunación diferentes a las observadas en las categorías educativas existentes, especialmente en la vacunación estacional. Además, el patrón de ausencia muestra una fuerte correlación con los valores faltantes de otras variables socioeconómicas y demográficas, como `marital_status`, `employment_status` y `rent_or_own`, lo que sugiere que la ausencia de datos no es completamente aleatoria.

Debido a que los valores faltantes parecen contener información relevante y forman parte de un patrón de no respuesta más amplio, se decidió conservar dicha información mediante la creación de una categoría específica `Unknown`.

In [38]:
df['education'] = df['education'].fillna('Unknown')
df['education'].value_counts(dropna=False)

education
College Graduate    10097
Some College         7043
12 Years             5797
< 12 Years           2363
Unknown              1407
Name: count, dtype: int64

#### `employment_status`,  `rent_or_own`,  `marital_status`

In [42]:
print(df['employment_status'].value_counts(dropna=False))
print('----------------')
print(df['rent_or_own'].value_counts(dropna=False))
print('----------------')
print(df['marital_status'].value_counts(dropna=False))

employment_status
Employed              13560
Not in Labor Force    10231
NaN                    1463
Unemployed             1453
Name: count, dtype: int64
----------------
rent_or_own
Own     18736
Rent     5929
NaN      2042
Name: count, dtype: int64
----------------
marital_status
Married        13555
Not Married    11744
NaN             1408
Name: count, dtype: int64


In [43]:
pd.crosstab(
    df['employment_status'].isna(),
    df['marital_status'].isna()
)

marital_status,False,True
employment_status,,
False,25103,141
True,196,1267


In [44]:
pd.crosstab(
    df['employment_status'].isna(),
    df['rent_or_own'].isna()
)

rent_or_own,False,True
employment_status,,
False,24558,686
True,107,1356


In [45]:
pd.crosstab(
    df['marital_status'].isna(),
    df['rent_or_own'].isna()
)

rent_or_own,False,True
marital_status,,
False,24542,757
True,123,1285


Analizando los datos se refuerza la hipotesis de que una seccion demografica no responde parcialmente a la encuesta. 

`employment_status`

In [46]:
pd.crosstab(
    df['employment_status'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
employment_status,,
Employed,78.443953,21.556047
Not in Labor Force,78.076434,21.923566
Unemployed,83.688919,16.311081
Unknown,81.476418,18.523582


In [47]:
pd.crosstab(
    df['employment_status'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
employment_status,,
Employed,57.809735,42.190265
Not in Labor Force,44.208777,55.791223
Unemployed,69.786648,30.213352
Unknown,61.244019,38.755981


La variable employment_status presenta un 5,48% de valores faltantes. El análisis de las variables objetivo muestra diferencias relevantes en las tasas de vacunación entre las categorías existentes, especialmente para la vacuna estacional, donde los individuos clasificados como `Not in Labor Force` presentan tasas de vacunación considerablemente más altas que los grupos `Employed` y `Unemployed`.

Los registros con valores faltantes muestran un comportamiento diferenciado y, además, comparten un fuerte patrón de ausencia con otras variables demográficas como `education`, `marital_status` y `rent_or_own`. Esto sugiere que los valores faltantes no son completamente aleatorios y pueden contener información predictiva relevante.

Por este motivo, los valores faltantes fueron conservados como una categoría independiente (Unknown), preservando la información asociada al patrón de ausencia y evitando introducir supuestos mediante imputaciones tradicionales.

In [52]:
df['employment_status'] = df['employment_status'].fillna('Unknown')
df['employment_status'].value_counts(dropna=False)

employment_status
Employed              13560
Not in Labor Force    10231
Unknown                1463
Unemployed             1453
Name: count, dtype: int64

`rent_or_own`

In [48]:
pd.crosstab(
    df['rent_or_own'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
rent_or_own,,
Own,77.802092,22.197908
Rent,81.126666,18.873334
Unknown,80.607248,19.392752


In [53]:
pd.crosstab(
    df['rent_or_own'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
rent_or_own,,
Own,49.925278,50.074722
Rent,62.944847,37.055153
Unknown,58.080313,41.919687


La variable `rent_or_own` presenta un 7,65% de valores faltantes. El análisis de las tasas de vacunación muestra que los individuos con información desconocida presentan un comportamiento similar al grupo de personas que viven en viviendas de alquiler (Rent), especialmente en la vacunación contra la gripe H1N1. Sin embargo, se observan diferencias moderadas en la vacunación estacional, lo que sugiere que los valores faltantes no son completamente equivalentes a ninguna de las categorías existentes.

Dado que la ausencia de datos podría contener información adicional y forma parte de un patrón de no respuesta compartido con otras variables demográficas, los valores faltantes se conservaron como una categoría independiente (`Unknown`). Esta decisión permite preservar la información asociada al patrón de ausencia y evita introducir supuestos mediante una imputación directa a una categoría existente.

In [54]:
df['rent_or_own'] = df['rent_or_own'].fillna('Unknown')
df['rent_or_own'].value_counts(dropna=False)

rent_or_own
Own        18736
Rent        5929
Unknown     2042
Name: count, dtype: int64

`marital_status`

In [50]:
pd.crosstab(
    df['marital_status'].fillna('Unknown'),
    df['h1n1_vaccine'],
    normalize='index'
) * 100

h1n1_vaccine,0,1
marital_status,,
Married,76.628550,23.371450
Not Married,80.841281,19.158719
Unknown,81.818182,18.181818


In [51]:
pd.crosstab(
    df['marital_status'].fillna('Unknown'),
    df['seasonal_vaccine'],
    normalize='index'
) * 100

seasonal_vaccine,0,1
marital_status,,
Married,50.579122,49.420878
Not Married,55.824251,44.175749
Unknown,61.079545,38.920455


`marital_status` se parece bastante al caso de `rent_or_own`.Las diferencias entre *Not Married* y *Unknown* son muy pocas. Si el objetivo fuera únicamente la interpretación estadística, se podria defender fusionarlas. Sin embargo teniendo en cuenta las relaicones con otras variables la opcion más segura es crear una categoria *Unknown* para la variable. La razón es simple:

- No pierdes información.
- Mantienes consistencia con el tratamiento aplicado a otras variables demográficas.
- Los modelos basados en árboles decidirán si *Unknown* aporta señal o si debe comportarse como *Not Married*.

Recordemos que en los datos faltantes de `employment_status` el 92% también tienen faltantes en `rent_or_own` y el 87% también tienen datos faltantes en `marital_status` 

Eso indica que el missing no está ocurriendo de forma aislada. Probablemente existe un subconjunto de encuestados que omitió gran parte del bloque demográfico. Y esa información puede ser útil para el modelo.

La variable marital_status presenta un 5,27% de valores faltantes. El análisis de las tasas de vacunación muestra que los individuos con estado civil desconocido presentan un comportamiento similar al grupo *Not Married*, especialmente en la vacunación contra la gripe H1N1. Sin embargo, se observan diferencias más marcadas en la vacunación estacional y, además, los valores faltantes comparten un fuerte patrón de ausencia con otras variables demográficas como `education`, `employment_status` y `rent_or_own`.

Dado que este patrón sugiere que la ausencia de información puede contener señal predictiva y formar parte de un proceso de no respuesta más amplio, los valores faltantes se conservaron como una categoría independiente (*Unknown*). 